# 🧪 Финальная практика: LLM-судья на локальной Qwen3

**Задача:** написать собственный промпт, с помощью которого **локальная LLM** сравнивает ответ другой модели с эталоном, ставит **оценку 0–4**, а затем по выбранному порогу относит ответ к корректным или содержащим галлюцинацию.

**Что уже готово:** 30 размеченных примеров (встроены в ноутбук), запуск Qwen3 на GPU Google Colab, отправка запросов, проверка JSON, кэширование оценок, метрики и сохранение CSV.

**Что делают участники:** пишут промпт (**роль, задача, шкала, правила, JSON**), пробуют его на **трёх примерах из DEV**, затем проверяют **все 30**, выбирают порог по DEV и оценивают классификатор на TEST.

> **Без внешнего API для LLM и без ключей.** Для скачивания весов модели нужен интернет; сами запросы выполняются на GPU Colab. Это LLM-as-a-Judge, а **не** библиотека RAGAS. Эталон в упражнении считаем единственным источником истины.


## 1. Договоримся о разметке

- `llm_answer` — проверяемый ответ другой модели (**1-й столбец CSV**).
- `reference_answer` — эталонный ответ (**2-й столбец CSV**).
- `has_hallucination` — правильный класс (**3-й столбец CSV**): `1` = есть ошибка или неподтверждённый факт, `0` = нет.

**Что считать галлюцинацией в этой практике:** любое фактическое утверждение в `llm_answer`, которое **противоречит эталону** или **не подтверждается им**. Отсутствие некоторой информации из эталона не является галлюцинацией, если сказанное в ответе верно. Перефразирование допустимо.

Все 30 примеров синтетические: реальные знания о мире не нужны. В настоящих системах один краткий эталон часто не охватывает всех истинных фактов — этот эксперимент сознательно упрощён.

In [ ]:
# Установка в Google Colab (после установки при необходимости перезапустите среду)
%pip -q install "transformers==4.57.1" "accelerate>=1.6,<2" "bitsandbytes>=0.46" "pandas>=2,<3" "scikit-learn>=1.4" "matplotlib>=3.7" "tqdm>=4.65"

## 2. Данные: 30 примеров уже в ноутбуке

CSV из исходного задания встроен для удобства, поэтому отдельно загружать файл не нужно. При необходимости поставьте `USE_OWN_CSV = True`, чтобы проверить собственный CSV с колонками `llm_answer`, `reference_answer`, `has_hallucination`.

Правильная метка **не показывается локальной LLM**: она используется только в конце для подсчёта качества классификатора. Разделение на DEV и TEST фиксированное и стратифицированное.


In [ ]:
from io import BytesIO
from pathlib import Path
import base64
import gzip
import pandas as pd
from sklearn.model_selection import train_test_split

USE_OWN_CSV = False  # True: загрузить собственный CSV

# Встроенный CSV с 30 размеченными примерами (сжат для компактности ноутбука).
BUILTIN_CSV_GZIP_BASE64 = (
    "H4sIAL6o2moC/61ZzW4b1xXeF8g7XHiVoopAKqljyZsUbQ10lQJ9gEBwGFiALAdKgnpJ6seyIVmMHAMxbDdqnKKbbkYURxyRHBLw"
    "E9x5hb5AX6Hf+c65M3dISm6bAonFGd57/s93fviv0XRz8/5n61tf/bm1vbTd+qK13dq62wov7q1/9dm99c3Nb+5ubK1/vfFg671f"
    "+NOi7Sc+9cNix739h/8RH3s+LdpvR84PfOKnxW7xBC9z1/zI8bmNp1HRdSuNFbw5x+2+T5wf4sMYB3I8TvylEPsBZzM85G9Hy0vX"
    "cSo6xY5Pijb+neBl4kfXc1t2/nvhGQTs+KHzE9wWggdyf1aiZE6eBpR/hW/PcLEN5jv8m4EVTn6LR9AoOiJc205BRJDfcX7qJ86f"
    "FbugmxddP4b8rrG61mg40R2i4iMU/o+ok+4Qqh8KfaEk5CFzClqi+hE+Z9AlMb7ToouTeFc8KnZnODdvkXMTqr0Re+LQHm0dzCsM"
    "eyA2Ko5pJ1Hn185/509csU9FhdkI957Aykv/DyrOZxROfAyTTeVBHIo/PbhuX/UVPQfqfHCB3vj/Up30V3wcSKDgy7w49Kl6dgoW"
    "GfTHx4E4GLLsgEdX/TWAhBJWu2JdkUCPBvFgw3AcsibuQxowV6bv4LhHV46hXRvkSzLwkL/wfR5HqIphwJHqR1z1sAR3yU689UxS"
    "BqJmKjZod2iGMVwMVhNQzuHfEB14UzwpjopjeUyYVzQpje6YI7kGnhl1IpmjWvTp2Od6oVfsW/7qYUm34rGke3FIDzEkhScNfeaT"
    "tZjlkVvIDja4NGbqwmcqdl9CGscuEPA1vcQHYl4xGEx0pE6519r88pPNb+63tpZbD9fvf7nZKjWc0us9JkqmN6ZKR4yTimfcrz52"
    "q6urrtlofMD/oPUzKkNBcRcu0tRK5fUIwXmBrD6ck1U8uEv/Ct7kjiglATsUlgl8LlbruOIpwxb4RvPklOaS5IoDvDxcoNFtF8Jd"
    "4kruqPFKReRj6ih0RoX7VBhRpJHzE7RI4Ri9ByIpQydTGIVhkLhvqCsUINRqKuLfLr6XaLwJsxbHEp4Sff5lUNRnMNnf3kWQok2K"
    "x8WJH83ej0hrILxAYtbC+7AWBhI55+I8EDC3Ah5hvBOB3Z774+/uSMT9/uHd1iZk+yF8p8FaQkyIXglxOGVoEdxWhK15r1flM4FH"
    "WKhhXyoKM0joRjVbwsceo/uSujCPmV5jwuNj4gJZdXEQQghAKkho6gzFhq/UClKKbvjX8U2tjhmD83IR1oKr2O5oVqaJHy45S9YR"
    "C4hATtf9s/38CtY36BXUUorKrESWxxVPAI6a0YyZIBIKnjjYMc4zRT5hxzqLV7C+wHpf3V5RV6wI1CXRD4n3ERIj0s/EPaCWkj68"
    "u19V4AMIvj8DnS9hsjayT+qq+BX+TKxqg2qHd2giBRUzVcb4nUi8ZlFOyFkBGSkffHckQK8x2ofmVpauYjrnqVszV4mX2tT0CXoD"
    "+ZcpnJeFUd9k/HuMEgqdmT0SJql1BheKDdOqp4Ju9PMLiUUId4RQSIRmxmy8ZDC39cy3BKcE7r/BRkqSxlBRHZkFzAnEJ+L0uMNi"
    "I2R0ljQeYl4zsoBTU+vAkO5VnHzEgipZb2Fs7JG9zVuMK+YsLCEmr1+U1F7gtpl79G4ZRh2JcOb+VICiaCsovWZly6zXmlRwOyIy"
    "lfaI6vBuLe+yqmb9Zuvz7QcbnzttbZkxmWs2XdATtdVa4Vny821mTK+5UtY+5SkJgqeNT//kmjfnv9PciAHyqr5lao2OtghMl07o"
    "n9nRhOZhzErQhfGlgfixAnE5C99plBBGJbtSMXe9QyOKR4CeUv1A90BrwFXyqK9OCW4peexofxwcY8iYsDuyChfOWHmpuqxcG5uh"
    "UPb9srWWgLeBotiFln+ZOdavvjdorJeThWTcvNSKBT0C+bn2Xx3eaRuQi91LsJ8whDluSL+yHHIpjrGV5Q8V+gVqtfkGeknwn8Ag"
    "AiVdV7nLWqhMJyeRmt+di/WsAR4xn0i2r61fVcxKshJIrupUJ2WQGYywKUXoQfjc8vY4Yh2Gmpg5ge401DBHtZOYHioRrQ+SRD1p"
    "0tm1Wh4J6B/igtYbGy3LcJ9pEwjvVT21yqLQObBZZHxteJt81URxm13FDnGyTO6nvH6uTU5bo54dkla0uAeigHPwAgTFlHClovNi"
    "OAoprAbu41KzqD14t2k10L6X0NeSpB0a57X6DNVz1mF1cXRIBI3sXHJYq9I7IfZwuHxMuLEmJfp2phNhxZKmL+jVF48NgnSO7xKN"
    "W1sj5NHULIibs+QgM92dP9z51L2P8m0cJRJKedjKxvLot4bffvR29MswmkoFtz793Ia8UIlvYioX5FVwc8zgs5D6lIzNvBSpcx2V"
    "bvjvAo2quMVpJyQbQjaitSSBILAxFIhn1wAC0mO3SytkoQmUlUKPY+hRDOh4aRX6VVRfrRHJWUDMzc2bazIrT7UgTJgEEwIZty0C"
    "ArX4Vgy1nQzu1Ar/TANkg+D/wIOY8YK2F0TcE7vIrBwnexo6QJa/TqAmqY7X54yby+htxqlBbTnU/ocia8F6ZO/gmAGiQ5Jmn+3U"
    "aW12zGoNrjXM1R3HmKwJLRN2JGGmYDQjYWadZF1GvA19lppHJ/wO2aaavxHQZBq61QSdW96yWufagh8o/rHdj7YpsmwCyZMQI4lW"
    "aHQFI0lW28xRvp/HtLZJ6VuEVCu+mnZhxMyjka6KZGdI2taJjfBiAFsb0cIeRyqbVAcWuR12A8/pCNsIRdRk2qxRq4AxpHi1lApb"
    "FdNKMfbUCuSOImyAW+DTa05Pw3KiMfS1zjFoZvtFRuRTgsaFyPjG/10Xd3uMx4R72HJ4v4YXrbwfbxuU08oHdQ7G9zxaMhFOcpom"
    "mWOuLdyIw1/pJB1JJ6TFleh0Pod0x9gI34UpS+vrZbmDqgEDnm8bjAxslyxwsOuYSWMODbB7T5sdjlzGqWwVZHsk/fo7Ra41htq+"
    "y3WReKFoOiIvVAb94k+RUOX4UQqtU2xaDpJkO9YuUXHUgmpmXSz9lbMBTkyrpuMdoqWoU46BcUoA9lfXmlygXXl5upDZtSRDDsgU"
    "3mxEyWZLQ+uOuaVhOxu2GmVZY+OjVDia2o8HMnKtNKxvLVtqGdO11AGmn/3XFFc+aigyjWzGGi5ZVtui/qTChlzRqiefdZ81hRl0"
    "syiiGUzPVyw+ct0WhjQqn5s8OmlY8LA54qqM6YVXv723/eC+LEndnY3t1hcPHtrYXP0IUk4YKXu1VMfmubrZmydlSulvKlPZPesl"
    "Fr4wmkmwy4mcCK3mpA92QyATYbk8OdKuqb+gzeFvK2NCjK01O5RPzLKnP4ZIkMhPSLNvDYoXChLQrirHM5vARftbZkwmLr5dk0tb"
    "1HICK9dXV+WkAv2/AYwVFI2hGwAA"
)

if USE_OWN_CSV:
    CSV_PATH = Path("final_llm_judge_30_cases.csv")
    if not CSV_PATH.exists():
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("CSV не загружен")
        CSV_PATH = Path(next(iter(uploaded)))
    df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
else:
    csv_content = gzip.decompress(base64.b64decode(BUILTIN_CSV_GZIP_BASE64))
    df = pd.read_csv(BytesIO(csv_content), encoding="utf-8-sig")

required = {"llm_answer", "reference_answer", "has_hallucination"}
assert required.issubset(df.columns), f"Не хватает колонок: {required - set(df.columns)}"
assert df["has_hallucination"].isin([0, 1]).all(), "Метки должны быть 0 или 1"
assert df[["llm_answer", "reference_answer"]].notna().all().all(), "Есть пустые ответы"
df["case_id"] = range(1, len(df) + 1)

DEV_IDS, TEST_IDS = train_test_split(
    df["case_id"], test_size=0.6, random_state=42,
    stratify=df["has_hallucination"]
)
print(f"Всего примеров: {len(df)}; DEV: {len(DEV_IDS)}; TEST: {len(TEST_IDS)}")
# Здесь метку не выводим и не передаём судье.
display(df[["case_id", "llm_answer", "reference_answer"]].head(5))


## 3. Загружаем локальную модель-судью

Используем [`Qwen/Qwen3-4B-Instruct-2507`](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507), в **4-битной NF4-квантовке**. Рекомендуемое оборудование: **Google Colab → Среда выполнения → Сменить среду выполнения → T4 GPU** (или более мощная GPU).

Загрузка весов происходит один раз. API-ключи не нужны; генерация ответов идёт на выделенной GPU. В Colab модель может работать заметно медленнее облачных моделей, особенно на всех 30 примерах.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("Подключите GPU: Colab → Среда выполнения → Сменить среду выполнения → T4 GPU")

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MAX_NEW_TOKENS = 420  # Максимальная длина JSON-ответа, можно увеличить при обрезании
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("GPU:", torch.cuda.get_device_name(0))

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
)
model.eval()
print("Локальная модель готова:", MODEL_ID)


## 4. ⭐ ВАШЕ ЗАДАНИЕ: напишите промпт

Ориентир — слайд с промптом Fact-Checker: **роль**, **задача**, **шкала 0–4**, **правила сравнения** и **строгий JSON**.

Продумайте: как оценивать неверные числа? Как отличать грубую ошибку от мелкой? Что делать с фактами, которых нет в эталоне? Нужно ли наказывать за неполный, но верный ответ? Чем оценки `3` и `4` отличаются?

В ответе обязательно ожидаются следующие поля:

```json
{
  "accuracy_score": 2,
  "verdict": "Mixed",
  "error_type": "Unsupported information",
  "fact_check_details": "Что не совпало с эталоном",
  "reasoning": "Почему выбрана такая оценка",
  "correction": "Как исправить ответ, если нужно"
}
```

Это **образец структуры**, а не готовый промпт и не правильная оценка конкретного примера. **Оценка 4 — максимально достоверный ответ, 0 — самый проблемный.**

In [7]:
# ✍️ ИЗМЕНЯТЬ ТОЛЬКО ЭТУ ЯЧЕЙКУ
STUDENT_PROMPT = """
РОЛЬ: [Напишите, кем является модель-судья]

ЗАДАЧА: [Что и на основании чего нужно проверить?]

ШКАЛА (0–4):
0 — [ваши критерии]
1 — [ваши критерии]
2 — [ваши критерии]
3 — [ваши критерии]
4 — [ваши критерии]

ПРАВИЛА: [Что делать с перефразированиями, пропусками и новыми фактами?]

ФОРМАТ: [Опишите строгий JSON с полями
accuracy_score, verdict, error_type, fact_check_details, reasoning, correction]
"""

## 5. Сервисный код — менять не нужно

Функция `ask_model()` использует **только локальную Qwen3**: она не обращается к API облачных LLM. Для каждого примера модель получает **только эталонный ответ, ответ LLM и ваш промпт**, но **не правильную метку** `has_hallucination`.

Кэш работает в пределах сессии: уже оценённые три примера не пересчитываются при переходе к 30 строкам, пока вы не поменяли промпт или модель. Некорректный JSON учитывается как техническая ошибка, а не подменяется выдуманной оценкой.


In [8]:
import hashlib
import json
import re
from tqdm.auto import tqdm

REQUIRED_JSON_KEYS = {
    "accuracy_score", "verdict", "error_type",
    "fact_check_details", "reasoning", "correction",
}
CACHE = {}

def check_student_prompt():
    placeholders = ["[Напишите", "[Что и на основании", "[ваши критерии]", "[Опишите строгий JSON"]
    if any(x in STUDENT_PROMPT for x in placeholders):
        raise ValueError("Сначала заполните STUDENT_PROMPT в ячейке с заданием")
    if len(STUDENT_PROMPT.strip()) < 120:
        raise ValueError("Промпт слишком короткий: опишите роль, задачу, шкалу 0–4 и JSON")

def ask_model(prompt_text):
    """Локальная генерация: сеть к провайдеру LLM не используется."""
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([formatted], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][model_inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def parse_judge_json(text):
    clean = text.strip()
    clean = re.sub(r"^```(?:json)?\s*|\s*```$", "", clean, flags=re.I).strip()
    # Иногда локальная модель добавляет пояснение до/после JSON.
    if not clean.startswith("{") or not clean.endswith("}"):
        found = re.search(r"\{.*\}", clean, flags=re.S)
        if found:
            clean = found.group(0)
    obj = json.loads(clean)
    if not isinstance(obj, dict):
        raise ValueError("Ожидается JSON-объект")
    missing = REQUIRED_JSON_KEYS - set(obj)
    if missing:
        raise ValueError(f"Нет полей JSON: {sorted(missing)}")
    score = obj["accuracy_score"]
    if type(score) is not int or not 0 <= score <= 4:
        raise ValueError("accuracy_score должен быть целым числом 0..4")
    return obj

def evaluate_one(row):
    check_student_prompt()
    prompt = (
        STUDENT_PROMPT.strip()
        + "\n\n=== ЭТАЛОННЫЙ ОТВЕТ ===\n" + str(row.reference_answer)
        + "\n\n=== ОТВЕТ LLM ===\n" + str(row.llm_answer)
        + "\n\nВерни только JSON в заданном формате."
    )
    fingerprint = hashlib.sha256((MODEL_ID + str(MAX_NEW_TOKENS) + prompt).encode()).hexdigest()
    if fingerprint in CACHE:
        return {**CACHE[fingerprint], "case_id": int(row.case_id)}

    raw = None
    try:
        raw = ask_model(prompt)
        parsed = parse_judge_json(raw)
        record = {
            "case_id": int(row.case_id),
            "accuracy_score": parsed["accuracy_score"],
            "verdict": parsed["verdict"], "error_type": parsed["error_type"],
            "fact_check_details": parsed["fact_check_details"],
            "reasoning": parsed["reasoning"], "correction": parsed["correction"],
            "raw_answer": raw, "generation_error": None,
        }
        CACHE[fingerprint] = record.copy()
    except Exception as e:
        # Ошибки не кэшируем: можно поправить промпт и повторить запуск.
        record = {
            "case_id": int(row.case_id), "accuracy_score": None,
            "verdict": None, "error_type": None,
            "fact_check_details": None, "reasoning": None,
            "correction": None, "raw_answer": raw,
            "generation_error": f"{type(e).__name__}: {e}",
        }
    return record

def evaluate_many(frame):
    results = []
    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc="Локальный LLM-судья"):
        results.append(evaluate_one(row))
    return pd.DataFrame(results)


## 6. Первые три примера — проверка промпта

Сначала проверяем **три примера из DEV**. Смотрите, возвращается ли валидный JSON, насколько обоснованны оценки и объяснения. Если JSON неправильный — уточните промпт и повторите запуск.

Правильные метки этих примеров не показываются модели; их можно использовать для обсуждения после эксперимента. **TEST не используем для настройки промпта и порога.**


In [ ]:
# Первые три из DEV, чтобы не подглядывать в финальный TEST
preview_df = df[df.case_id.isin(DEV_IDS)].head(3).copy()
print("Проверяем case_id:", preview_df.case_id.to_list())
preview_scores = evaluate_many(preview_df)
preview_view = preview_df.merge(preview_scores, on="case_id", how="left")
display(preview_view[[
    "case_id", "llm_answer", "reference_answer",
    "accuracy_score", "reasoning", "generation_error"
]])

if preview_scores["accuracy_score"].notna().sum() == 0:
    print("⚠️ Все три ответа не удалось разобрать. Посмотрите generation_error и raw_answer:")
    display(preview_scores[["case_id", "generation_error", "raw_answer"]])
else:
    print("Успешных оценок:", int(preview_scores["accuracy_score"].notna().sum()), "/ 3")


## 7. Оценка всех 30 примеров

Теперь используем тот же промпт для всех строк. Оценённые ранее примеры возьмутся из кэша, **если вы не меняли промпт**. На T4 локальная модель может обрабатывать все 30 вопросов заметно дольше, чем облачная.

В случае ошибки формата смотрите `generation_error` и `raw_answer`: вывод модели сохраняется для диагностики.


In [ ]:
all_scores = evaluate_many(df)
results = df.merge(all_scores, on="case_id", how="left", validate="one_to_one")
print(f"Оценок получено: {results.accuracy_score.notna().sum()} из {len(results)}")
display(results[["case_id", "llm_answer", "accuracy_score", "reasoning", "generation_error"]].head(10))
if results.accuracy_score.notna().sum() == 0:
    raise RuntimeError("Нет ни одной корректной оценки. Исправьте промпт / подключение.")

## 8. Подбираем порог только по DEV

Шкала: **чем выше `accuracy_score`, тем надёжнее ответ**. Для порога `THRESHOLD` оценки **ниже порога** означают «галлюцинация» (`1`), оценки **не ниже порога** — «ответ корректный» (`0`).

Подбираем порог **на DEV (12 строк)** и затем оцениваем на **TEST (18 строк)**, не используя TEST для выбора промпта и порога. Для учебного анализа покажем показатели и на всех 30 примерах.


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, balanced_accuracy_score,
)

dev = results[results.case_id.isin(DEV_IDS)].copy()
test = results[results.case_id.isin(TEST_IDS)].copy()

# 1 означает ГАЛЛЮЦИНАЦИЮ — именно её precision/recall мы считаем.
def quality_at_threshold(frame, threshold):
    valid = frame.dropna(subset=["accuracy_score"]).copy()
    if valid.empty:
        return {"n": 0, "coverage": 0.0, "accuracy": float("nan"),
                "precision": float("nan"), "recall": float("nan"),
                "f1": float("nan"), "balanced_accuracy": float("nan")}
    pred = (valid.accuracy_score < threshold).astype(int)
    true = valid.has_hallucination.astype(int)
    return {
        "n": len(valid), "coverage": len(valid) / len(frame),
        "accuracy": accuracy_score(true, pred),
        "precision": precision_score(true, pred, zero_division=0),
        "recall": recall_score(true, pred, zero_division=0),
        "f1": f1_score(true, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(true, pred),
    }

thresholds = []
for t in [1, 2, 3, 4]:
    thresholds.append({"threshold": t, **quality_at_threshold(dev, t)})
threshold_table = pd.DataFrame(thresholds)
print("Метрики только на DEV (12 примеров):")
display(threshold_table.round(3))

Метрики только на DEV (12 примеров):


,threshold,n,coverage,accuracy,precision,recall,f1,balanced_accuracy
0,1,12,1.0,0.500,0.0,0.000,0.000,0.500
1,2,12,1.0,0.917,1.0,0.833,0.909,0.917
2,3,12,1.0,1.000,1.0,1.000,1.000,1.000
3,4,12,1.0,0.500,0.5,1.000,0.667,0.500


## 9. ⭐ Ваш выбор: порог принятия ответа

**Выберите порог сами.** Например, `THRESHOLD = 3` означает, что оценки 3 и 4 — «OK», оценки 0–2 — «галлюцинация». Можно выбрать 1, 2, 3 или 4; объясните свой выбор через баланс **Precision / Recall**.

Если **опаснее пропустить галлюцинацию**, нужен высокий Recall. Если **опаснее зря забраковать правильный ответ**, нужен высокий Precision. Не подбирайте порог по TEST.

In [12]:
# ✍️ ИЗМЕНИТЬ: выберите порог по таблице DEV выше
THRESHOLD = 3
assert THRESHOLD in [1, 2, 3, 4]

## 10. Финальная оценка качества классификатора

Ниже метрики на **TEST** и **всех 30 примерах**. Отсутствующий или невалидный JSON **не считается правильной классификацией**: его доля отдельно показана как `coverage`. Другие метрики вычисляются **только по строкам, где ответ судьи удалось разобрать**. В реальном проекте технические ошибки тоже нужно контролировать.

In [ ]:
quality_test = quality_at_threshold(test, THRESHOLD)
quality_all = quality_at_threshold(results, THRESHOLD)
quality_comparison = pd.DataFrame([
    {"sample": "TEST (18)", **quality_test},
    {"sample": "ALL (30)", **quality_all},
])
display(quality_comparison.round(3))

valid_test = test.dropna(subset=["accuracy_score"]).copy()
if len(valid_test):
    y_true = valid_test["has_hallucination"].astype(int)
    y_pred = (valid_test["accuracy_score"] < THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"Confusion matrix для TEST: TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print("TP — поймали галлюцинацию; FN — пропустили галлюцинацию")
    print("FP — ошибочно забраковали верный ответ; TN — приняли верный ответ")
    mistakes = valid_test.loc[y_true.to_numpy() != y_pred.to_numpy()].copy()
    mistakes["predicted_hallucination"] = (mistakes.accuracy_score < THRESHOLD).astype(int)
    display(mistakes[["case_id", "llm_answer", "reference_answer", "has_hallucination",
                     "accuracy_score", "predicted_hallucination", "reasoning"]])
else:
    print("На TEST нет распознанных оценок — проверьте API и JSON.")

## 11. Визуализация и выгрузка результатов

Сравните, как меняется полнота обнаружения галлюцинаций и точность с изменением порога. CSV с предсказаниями можно разобрать вместе с группой.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
for metric in ["precision", "recall", "f1"]:
    ax.plot(threshold_table["threshold"], threshold_table[metric], marker="o", label=metric)
ax.set(xlabel="Порог (score ниже порога = галлюцинация)", ylabel="Значение метрики", ylim=(-0.04, 1.04))
ax.set_xticks([1, 2, 3, 4])
ax.legend()
ax.grid(alpha=0.25)
plt.show()

results["predicted_hallucination"] = pd.NA
mask = results.accuracy_score.notna()
results.loc[mask, "predicted_hallucination"] = (
    results.loc[mask, "accuracy_score"] < THRESHOLD
).astype(int)
results["sample"] = results.case_id.map(lambda x: "DEV" if x in set(DEV_IDS) else "TEST")
RESULT_CSV = "llm_judge_predictions.csv"
results.to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")
print("Сохранено:", RESULT_CSV)
try:
    from google.colab import files
    files.download(RESULT_CSV)
except ImportError:
    pass

## 12. Что означают метрики

Считаем **положительным классом наличие галлюцинации** (`1`).

| Метрика | Смысл | На что обращать внимание |
|---|---|---|
| **Accuracy** | `(TP + TN) / (TP + TN + FP + FN)` | Доля всех верных решений; может обманывать на несбалансированных данных. |
| **Precision** | `TP / (TP + FP)` | Среди найденных галлюцинаций сколько действительно ошибочные? |
| **Recall / Полнота** | `TP / (TP + FN)` | Какую долю реальных галлюцинаций мы поймали? |
| **F1** | `2 × Precision × Recall / (Precision + Recall)` | Баланс Precision и Recall. |
| **Balanced accuracy** | Среднее Recall для класса 0 и класса 1 | Удобно, когда классы не сбалансированы. |
| **Coverage** | Доля примеров с корректным JSON и оценкой 0–4 | Показывает устойчивость промпта и локальной модели. |

**TP:** галлюцинация была — судья её обнаружил. **FN:** галлюцинация была — судья пропустил. **FP:** ответ правильный — судья ошибочно отклонил. **TN:** правильный ответ принят.

> **Важно:** высокая оценка самого LLM-судьи (`accuracy_score=4`) — это *не* метрика Accuracy. Оценку 0–4 ставит модель; Accuracy вычисляется позже **по размеченному CSV**.

### Вопросы для обсуждения

1. Где судья ошибался чаще: на числах, на неполных ответах или на дополнительных деталях?
2. При каком пороге меньше **пропущенных галлюцинаций**? Сколько при этом появилось ложных срабатываний?
3. Почему судья может ошибаться, даже если эталон выглядит однозначным?
4. Как изменится F1, если поменять описание роли, задачи, критериев оценки или требование к JSON?